# Lab 8 — Topic Modeling (LSA / LDA)

Track A corpus exploration on the cleaned `processed_v2` Ukrainian review/comment corpus.

## 1. Install deps

In [1]:
from pathlib import Path
import subprocess
import sys

if Path('/content').exists() and not Path('/content/nlp_labs').exists():
    subprocess.run(['git', 'clone', 'https://github.com/velotsuraptor/nlp_labs.git', '/content/nlp_labs'], check=True)

project_root_candidates = [
    Path('/content/nlp_labs/project_lab8'),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PROJECT_ROOT = None
for cand in project_root_candidates:
    if (cand / 'requirements.txt').exists() and (cand / 'src').exists():
        PROJECT_ROOT = cand
        break
    if (cand / 'project_lab8' / 'requirements.txt').exists():
        PROJECT_ROOT = cand / 'project_lab8'
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate project_lab8/requirements.txt')

REPO_ROOT = PROJECT_ROOT.parent
req_path = PROJECT_ROOT / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)], check=True)
print('project_root =', PROJECT_ROOT)

project_root = C:\Users\maia1\data\politiekh\masters\nlp\project_lab8


## 2. Data access

In [2]:
from pathlib import Path
import sys
import pandas as pd

ROOT = PROJECT_ROOT
sys.path.insert(0, str(REPO_ROOT))

from project_lab8.src.topic_utils import load_processed_corpus, prepare_topic_corpus
from project_lab8.src.topic_modeling import TopicModelConfig, fit_topic_model, topic_words_frame, top_documents_frame

raw_df = load_processed_corpus(ROOT)
raw_df.head(3)

,text_id,text,sentences,label
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...","[""Вступив на ІСТ цього року, тепер молюся, щоб...",Question / Request for Help
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[""Цифрова держава Повідомлення 123 від 18.04.2...",Question / Request for Help
2,3099,Старий університет поки що вчить. Наразі налаш...,"[""Старий університет поки що вчить."", ""Наразі ...",Neutral Comment


## 3. Corpus filtering / preprocessing checks

In [3]:
corpus_df = prepare_topic_corpus(raw_df, text_col='text', min_tokens=4)
print('documents before filtering:', len(raw_df))
print('documents after filtering:', len(corpus_df))
print('removed:', len(raw_df) - len(corpus_df))
corpus_df[['text_id', 'label', 'topic_token_count', 'topic_text']].head(5)

documents before filtering: 1000
documents after filtering: 988
removed: 12


,text_id,label,topic_token_count,topic_text
0,9905,Question / Request for Help,11,вступив іст тепер молюся щоб переклали пуа адж...
1,10001201,Question / Request for Help,20,цифрова держава повідомлення надіслано згодом ...
2,3099,Neutral Comment,10,старий університет вчить наразі налаштований п...
3,8664,Gratitude / Positive Feedback,7,ринок цнап львова швидке якісне вирішення питань
4,1035,Suggestion / Idea,6,здається наша кузня супер-кадрів виглядати суч...


## 4. Vectorizer setup

In [4]:
configs = [
    TopicModelConfig(name='lsa_k5', model_type='lsa', vectorizer_type='tfidf', n_topics=5, ngram_range=(1, 2), min_df=5, max_df=0.8),
    TopicModelConfig(name='lsa_k8', model_type='lsa', vectorizer_type='tfidf', n_topics=8, ngram_range=(1, 2), min_df=5, max_df=0.8),
    TopicModelConfig(name='lda_k5', model_type='lda', vectorizer_type='count', n_topics=5, ngram_range=(1, 1), min_df=5, max_df=0.8),
    TopicModelConfig(name='lda_k8', model_type='lda', vectorizer_type='count', n_topics=8, ngram_range=(1, 1), min_df=5, max_df=0.8),
]

pd.DataFrame([
    {
        'name': cfg.name,
        'model_type': cfg.model_type,
        'vectorizer_type': cfg.vectorizer_type,
        'n_topics': cfg.n_topics,
        'ngram_range': str(cfg.ngram_range),
        'min_df': cfg.min_df,
        'max_df': cfg.max_df,
    }
    for cfg in configs
])

,name,model_type,vectorizer_type,n_topics,ngram_range,min_df,max_df
0,lsa_k5,lsa,tfidf,5,"(1, 2)",5,0.8
1,lsa_k8,lsa,tfidf,8,"(1, 2)",5,0.8
2,lda_k5,lda,count,5,"(1, 1)",5,0.8
3,lda_k8,lda,count,8,"(1, 1)",5,0.8


## 5. LSA experiments

In [5]:
results = {}
for cfg in configs:
    results[cfg.name] = fit_topic_model(corpus_df['topic_text'], cfg)

lsa_word_tables = {name: topic_words_frame(res) for name, res in results.items() if name.startswith('lsa_')}
for name, table in lsa_word_tables.items():
    print(f'\n{name}')
    display(table)


lsa_k5


,model,topic_id,top_words
0,lsa_k5,0,"місце, щоб, обов, язково, обов язково, підняти..."
1,lsa_k5,1,"євідновлення, через, щоб, при, заяву, документ..."
2,lsa_k5,2,"будівля, університет, красива, час, міста, зна..."
3,lsa_k5,3,"місце, університет, язково, обов язково, обов,..."
4,lsa_k5,4,"будівля, євідновлення, університет, красива, з..."



lsa_k8


,model,topic_id,top_words
0,lsa_k8,0,"місце, щоб, обов, обов язково, язково, підняти..."
1,lsa_k8,1,"євідновлення, через, щоб, при, заяву, документ..."
2,lsa_k8,2,"будівля, університет, красива, працюють, знахо..."
3,lsa_k8,3,"місце, обов язково, язково, обов, гарне, уніве..."
4,lsa_k8,4,"будівля, євідновлення, університет, красива, з..."
5,lsa_k8,5,"університет, щоб, хороший, імені, відвідати, у..."
6,lsa_k8,6,"юриста, юриста олега, олега, питань, консульта..."
7,lsa_k8,7,"євідновлення, щоб, час, послуги, роботи, уніве..."


## 6. LDA experiments

In [6]:
lda_word_tables = {name: topic_words_frame(res) for name, res in results.items() if name.startswith('lda_')}
for name, table in lda_word_tables.items():
    print(f'\n{name}')
    display(table)


lda_k5


,model,topic_id,top_words
0,lda_k5,0,"через, заклад, людей, консультацію, академії, ..."
1,lda_k5,1,"євідновлення, заяву, дія, персонал, майно, щоб..."
2,lda_k5,2,"при, університет, області, щоб, роботу, україн..."
3,lda_k5,3,"піднятися, обов, щоб, язково, варто, місто, мі..."
4,lda_k5,4,"будівля, документи, немає, робити, паспорт, на..."



lda_k8


,model,topic_id,top_words
0,lda_k8,0,"через, може, завжди, працюють, дію, заклад, бі..."
1,lda_k8,1,"євідновлення, заяву, дія, щоб, майно, консульт..."
2,lda_k8,2,"щоб, роботу, грн, області, при, роботи, україн..."
3,lda_k8,3,"язково, обов, університет, місце, центр, себе,..."
4,lda_k8,4,"робити, ремонт, будинок, навіть, через, євідно..."
5,lda_k8,5,"навчання, документи, викладачі, працівників, з..."
6,lda_k8,6,"піднятися, міста, будівля, місто, варто, вид, ..."
7,lda_k8,7,"місце, обслуговування, номер, людей, враження,..."


## 7. Top words per topic

In [7]:
topic_words_all = pd.concat([topic_words_frame(res) for res in results.values()], ignore_index=True)
topic_words_all

,model,topic_id,top_words
0,lsa_k5,0,"місце, щоб, обов, язково, обов язково, підняти..."
1,lsa_k5,1,"євідновлення, через, щоб, при, заяву, документ..."
2,lsa_k5,2,"будівля, університет, красива, час, міста, зна..."
3,lsa_k5,3,"місце, університет, язково, обов язково, обов,..."
4,lsa_k5,4,"будівля, євідновлення, університет, красива, з..."
5,lsa_k8,0,"місце, щоб, обов, обов язково, язково, підняти..."
6,lsa_k8,1,"євідновлення, через, щоб, при, заяву, документ..."
7,lsa_k8,2,"будівля, університет, красива, працюють, знахо..."
8,lsa_k8,3,"місце, обов язково, язково, обов, гарне, уніве..."
9,lsa_k8,4,"будівля, євідновлення, університет, красива, з..."


## 8. Top documents per topic

In [8]:
top_docs_all = pd.concat([top_documents_frame(res, corpus_df, top_n=2) for res in results.values()], ignore_index=True)
for model_name in ['lsa_k5', 'lsa_k8', 'lda_k5', 'lda_k8']:
    print(f'\nTop documents for {model_name}')
    display(top_docs_all[top_docs_all['model'] == model_name])


Top documents for lsa_k5


,model,topic_id,rank,text_id,label,topic_score,excerpt
0,lsa_k5,0,1,14799,Suggestion / Idea,0.437354,Якщо ви у Львові - вам обов'язково варто підня...
1,lsa_k5,0,2,15063,Suggestion / Idea,0.418233,Найкрасивіші види на місто. Обов'язково варто ...
2,lsa_k5,1,1,10004398,Question / Request for Help,0.348064,♦️Як отримати допомогу? 1. Повідомити про пошк...
3,lsa_k5,1,2,10005388,Question / Request for Help,0.314137,"Доброго дня. Чи є тут люди, в яких пошкоджений..."
4,lsa_k5,2,1,3720,Neutral Comment,0.373368,Красива будівля зараз на реставрації. Усередин...
5,lsa_k5,2,2,16859,Neutral Comment,0.357253,Не всіма улюблене міське управління. але будів...
6,lsa_k5,3,1,365,Question / Request for Help,0.422902,Дуже чисте та екологічне місце. Де можна добре...
7,lsa_k5,3,2,16072,Suggestion / Idea,0.394846,Дуже гарне місце. Обов'язково беріть екскурсов...
8,lsa_k5,4,1,1942,Neutral Comment,0.355368,"Красива будівля знаходиться в центрі міста, по..."
9,lsa_k5,4,2,16859,Neutral Comment,0.345195,Не всіма улюблене міське управління. але будів...



Top documents for lsa_k8


,model,topic_id,rank,text_id,label,topic_score,excerpt
10,lsa_k8,0,1,14799,Suggestion / Idea,0.437460,Якщо ви у Львові - вам обов'язково варто підня...
11,lsa_k8,0,2,15063,Suggestion / Idea,0.419181,Найкрасивіші види на місто. Обов'язково варто ...
12,lsa_k8,1,1,10004398,Question / Request for Help,0.353913,♦️Як отримати допомогу? 1. Повідомити про пошк...
13,lsa_k8,1,2,10005388,Question / Request for Help,0.311044,"Доброго дня. Чи є тут люди, в яких пошкоджений..."
14,lsa_k8,2,1,3720,Neutral Comment,0.373338,Красива будівля зараз на реставрації. Усередин...
15,lsa_k8,2,2,11360,Suggestion / Idea,0.371158,"Будівля красива, а от людям, які в ній працюют..."
16,lsa_k8,3,1,365,Question / Request for Help,0.490792,Дуже чисте та екологічне місце. Де можна добре...
17,lsa_k8,3,2,16550,Suggestion / Idea,0.456658,Дуже гарне місце! Обов'язково додайте його до ...
18,lsa_k8,4,1,1942,Neutral Comment,0.340014,"Красива будівля знаходиться в центрі міста, по..."
19,lsa_k8,4,2,16859,Neutral Comment,0.339307,Не всіма улюблене міське управління. але будів...



Top documents for lda_k5


,model,topic_id,rank,text_id,label,topic_score,excerpt
26,lda_k5,0,1,10001589,Suggestion / Idea,0.982308,Сьогодні - рівно 5 місяців повномасштабної вій...
27,lda_k5,0,2,8559,Complaint / Dissatisfaction,0.945237,Робота вкрай неефективна. Сервіс онлайн реєстр...
28,lda_k5,1,1,10002246,Question / Request for Help,0.970462,Боже мій Всемогутній! Ти оберігаєш невинних та...
29,lda_k5,1,2,10001228,Question / Request for Help,0.966357,"Зверніть увагу, подати ""Повідомлення про пошко..."
30,lda_k5,2,1,5106,Gratitude / Positive Feedback,0.949552,"Комп'ютерна Академія ""КРОК"" одна з найкращих К..."
31,lda_k5,2,2,987,Gratitude / Positive Feedback,0.932611,"Чудовий університет. Дають добрі знання, які п..."
32,lda_k5,3,1,14514,Suggestion / Idea,0.955290,"Обов'язково варто піднятися на башту ратуші, в..."
33,lda_k5,3,2,14868,Suggestion / Idea,0.954735,"Раджу піднятися на вежу ратуші, дуже гарний ви..."
34,lda_k5,4,1,10004373,Question / Request for Help,0.957359,Всі коментарі стосовно допомоги дуже вірні і в...
35,lda_k5,4,2,10000639,Suggestion / Idea,0.956514,"попит буде високим, бо в такій квартирі можна ..."



Top documents for lda_k8


,model,topic_id,rank,text_id,label,topic_score,excerpt
36,lda_k8,0,1,10001589,Suggestion / Idea,0.980956,Сьогодні - рівно 5 місяців повномасштабної вій...
37,lda_k8,0,2,8559,Complaint / Dissatisfaction,0.941613,Робота вкрай неефективна. Сервіс онлайн реєстр...
38,lda_k8,1,1,10001228,Question / Request for Help,0.963524,"Зверніть увагу, подати ""Повідомлення про пошко..."
39,lda_k8,1,2,10004638,Question / Request for Help,0.958295,"? Документи, опитування, шеринг авто, оплата ш..."
40,lda_k8,2,1,10005801,Question / Request for Help,0.932631,А можна все ж таки користуватися мінімальним н...
41,lda_k8,2,2,8230,Neutral Comment,0.927021,Личаківський відділ соціального захисту змінив...
42,lda_k8,3,1,3825,Complaint / Dissatisfaction,0.937416,"Напевно, був би кращім місцем, якби не було ко..."
43,lda_k8,3,2,6099,Neutral Comment,0.926990,Красноградський районий суд Харківської област...
44,lda_k8,4,1,10004373,Question / Request for Help,0.953913,Всі коментарі стосовно допомоги дуже вірні і в...
45,lda_k8,4,2,10000639,Suggestion / Idea,0.953877,"попит буде високим, бо в такій квартирі можна ..."


## 9. Manual interpretation

In [9]:
MANUAL_TOPICS = {
    'lsa_k8::1': {
        'title': 'єВідновлення / заява через Дію',
        'explanation': 'Тема тримається на словах «євідновлення», «заяву», «майно», «дія». Топ-документи — інструктивні дописи про подання повідомлення про пошкоджене майно та роботу програми. Для цього корпусу це одна з найпредметніших тем LSA.',
    },
    'lsa_k8::6': {
        'title': 'Правові консультації / рекомендації юриста',
        'explanation': 'Слова «юриста», «олега», «питань», «консультацію» формують вузьку, але читабельну тему. Топ-документи прямо рекомендують конкретного юриста для сімейних і земельних питань. Це радше нішевий сюжет, ніж глобальна тема корпусу, але інтерпретується добре.',
    },
    'lda_k8::1': {
        'title': 'єВідновлення / документи / цифрові сервіси',
        'explanation': 'Тема підтримується словами «євідновлення», «заяву», «дія», «майно», «подати». Топ-документи описують, як подати повідомлення про пошкоджене майно та які сервіси доступні через Дію. Це одна з найкращих тем у всій ЛР8, бо вона має і зрозумілі слова, і узгоджені документи.',
    },
    'lda_k8::5': {
        'title': 'Освітні заклади / навчання / викладачі',
        'explanation': 'Слова «навчання», «викладачі», «академії», «атмосфера» формують окремий освітній кластер. Топ-документи — відгуки про академії та університети, де домінує опис навчального процесу і персоналу. Це корисна тема, бо корпус справді містить багато оглядів навчальних закладів.',
    },
    'lda_k8::6': {
        'title': 'Ратуша / вид на місто / туристична рекомендація',
        'explanation': 'Тема дуже стабільна: «піднятися», «місто», «вид», «ратуші», «вежу». Топ-документи — короткі рекомендації піднятися на ратушу та подивитися на місто з висоти. Це найконкретніша туристична тема корпусу.',
    },
}

manual_topic_df = pd.DataFrame([
    {'topic_key': key, 'title': value['title'], 'explanation': value['explanation']}
    for key, value in MANUAL_TOPICS.items()
])
manual_topic_df

,topic_key,title,explanation
0,lsa_k8::1,єВідновлення / заява через Дію,"Тема тримається на словах «євідновлення», «зая..."
1,lsa_k8::6,Правові консультації / рекомендації юриста,"Слова «юриста», «олега», «питань», «консультац..."
2,lda_k8::1,єВідновлення / документи / цифрові сервіси,"Тема підтримується словами «євідновлення», «за..."
3,lda_k8::5,Освітні заклади / навчання / викладачі,"Слова «навчання», «викладачі», «академії», «ат..."
4,lda_k8::6,Ратуша / вид на місто / туристична рекомендація,"Тема дуже стабільна: «піднятися», «місто», «ви..."


## 10. “Bad topics” analysis

In [10]:
BAD_TOPICS = {
    'lsa_k8::0': {
        'problem': 'duplicate / style-heavy topic',
        'why': 'Тема зібрана навколо шаблону рекомендації «обов’язково варто піднятися» і майже дублює іншу sightseeing-тему. Вона ловить стиль поради, а не ширший зміст корпусу.',
        'next_fix': 'Посилити stop-word filtering для загальних рекомендаційних слів, збільшити min_df або обмежити корпус однією доменною підтемою.',
    },
    'lsa_k8::3': {
        'problem': 'too generic topic',
        'why': 'Тут змішані слова на кшталт «місце», «гарне», «працюють», «обслуговування». Вони не задають конкретний сюжет і зводять тему до загальної оцінної лексики.',
        'next_fix': 'Прибрати ще більше generic opinion words і сильніше фільтрувати короткі документи.',
    },
    'lda_k8::0': {
        'problem': 'mixed topic',
        'why': 'Топ-слова і топ-документи змішують сервісні звернення, загальні war/support повідомлення і оцінні тексти. Це ознака неоднорідного корпусу та надто широкого простору тем.',
        'next_fix': 'Розділити корпус за доменами або прибрати дуже загальні документи до побудови тем.',
    },
    'lda_k8::2': {
        'problem': 'generic service/documents topic',
        'why': 'Слова «роботу», «документи», «послуги», «області» надто загальні і не дають чіткої предметної назви. Топ-документи також розходяться між адмінпослугами та ширшими побутовими питаннями.',
        'next_fix': 'Спробувати lemma_text, підняти min_df і додати окремий список stopwords для службової лексики.',
    },
}

bad_topic_df = pd.DataFrame([
    {'topic_key': key, 'problem': value['problem'], 'why': value['why'], 'next_fix': value['next_fix']}
    for key, value in BAD_TOPICS.items()
])
bad_topic_df

,topic_key,problem,why,next_fix
0,lsa_k8::0,duplicate / style-heavy topic,Тема зібрана навколо шаблону рекомендації «обо...,Посилити stop-word filtering для загальних рек...
1,lsa_k8::3,too generic topic,"Тут змішані слова на кшталт «місце», «гарне», ...",Прибрати ще більше generic opinion words і сил...
2,lda_k8::0,mixed topic,Топ-слова і топ-документи змішують сервісні зв...,Розділити корпус за доменами або прибрати дуже...
3,lda_k8::2,generic service/documents topic,"Слова «роботу», «документи», «послуги», «облас...","Спробувати lemma_text, підняти min_df і додати..."


## 11. LSA vs LDA comparison

In [11]:
comparison_text = (
    'Для цього змішаного корпусу LDA виявився кориснішим за LSA. '
    'LDA дав кілька тем, які добре читаються і по словах, і по документах: єВідновлення / Дія, освітні заклади, туристичні рекомендації про ратушу. '
    'LSA теж знаходить осмислені сюжети, але частіше дублює найсильніший лексичний шаблон і гірше розводить близькі за стилем документи. '
    'Найбільша проблема обох моделей — неоднорідність корпусу: тут одночасно є відгуки про установи, освітні тексти, туризм, адміністративні послуги і воєнно-гуманітарні звернення. '
    'Через це частина тем неминуче виходить змішаною або надто загальною. '
    'Для цього кейсу topic modeling корисний як інструмент корпусного огляду, але не як фінальна чиста сегментація без додаткового доменного відбору.'
)
print(comparison_text)

Для цього змішаного корпусу LDA виявився кориснішим за LSA. LDA дав кілька тем, які добре читаються і по словах, і по документах: єВідновлення / Дія, освітні заклади, туристичні рекомендації про ратушу. LSA теж знаходить осмислені сюжети, але частіше дублює найсильніший лексичний шаблон і гірше розводить близькі за стилем документи. Найбільша проблема обох моделей — неоднорідність корпусу: тут одночасно є відгуки про установи, освітні тексти, туризм, адміністративні послуги і воєнно-гуманітарні звернення. Через це частина тем неминуче виходить змішаною або надто загальною. Для цього кейсу topic modeling корисний як інструмент корпусного огляду, але не як фінальна чиста сегментація без додаткового доменного відбору.


## 12. Generate docs/audit_summary_lab8.md

In [12]:
from pathlib import Path

docs_dir = ROOT / 'docs'
docs_dir.mkdir(parents=True, exist_ok=True)
(ROOT / 'labs' / 'lab08').mkdir(parents=True, exist_ok=True)

best_topics = [
    ('lda_k8::1', 'єВідновлення / документи / цифрові сервіси'),
    ('lda_k8::6', 'Ратуша / вид на місто / туристична рекомендація'),
    ('lda_k8::5', 'Освітні заклади / навчання / викладачі'),
]
worst_topics = [
    ('lsa_k8::0', 'duplicate sightseeing/style topic'),
    ('lsa_k8::3', 'generic opinion/place topic'),
    ('lda_k8::0', 'mixed war/service topic'),
]

summary_md = f'''# Audit summary — Lab8

1. Corpus size after filtering: {len(corpus_df)} documents (from {len(raw_df)} original rows).
2. Models tested: LSA (TF-IDF + TruncatedSVD) and LDA (CountVectorizer + LatentDirichletAllocation).
3. Topic counts tested: k=5 and k=8 for both models.
4. Best themes: {best_topics[0][1]}; {best_topics[1][1]}; {best_topics[2][1]}.
5. Worst themes: {worst_topics[0][1]}; {worst_topics[1][1]}; {worst_topics[2][1]}.
6. What damaged the weak topics: template-style recommendation phrases, generic service vocabulary, and a mixed multi-domain corpus with many short documents.
7. Better model for this corpus: LDA, because its strongest topics stay more coherent at the document level.
8. Next steps: stronger stop-word filtering, domain-specific subcorpora, and optional lemma-based topic modeling.
'''
(docs_dir / 'audit_summary_lab8.md').write_text(summary_md, encoding='utf-8')

topic_notes_lines = []
topic_notes_lines.append('# Topic notes — Lab8')
topic_notes_lines.append('')
topic_notes_lines.append('## 1. Models and parameters')
topic_notes_lines.append('')
for cfg in configs:
    topic_notes_lines.append(f'- `{cfg.name}`: model={cfg.model_type}, vectorizer={cfg.vectorizer_type}, k={cfg.n_topics}, ngram_range={cfg.ngram_range}, min_df={cfg.min_df}, max_df={cfg.max_df}')

topic_notes_lines.append('')
topic_notes_lines.append('## 2. Interpreted topics')
topic_notes_lines.append('')
for key, info in MANUAL_TOPICS.items():
    model_name, topic_id = key.split('::')
    topic_id = int(topic_id)
    words = topic_words_all[(topic_words_all['model'] == model_name) & (topic_words_all['topic_id'] == topic_id)]['top_words'].iloc[0]
    topic_notes_lines.append(f'### {key} — {info["title"]}')
    topic_notes_lines.append(f'- Top words: {words}')
    topic_notes_lines.append(f'- Explanation: {info["explanation"]}')
    doc_rows = top_docs_all[(top_docs_all['model'] == model_name) & (top_docs_all['topic_id'] == topic_id)].sort_values('rank')
    for _, doc_row in doc_rows.iterrows():
        topic_notes_lines.append(f'- Top doc {int(doc_row["rank"])}: text_id={int(doc_row["text_id"])}, label={doc_row["label"]}, excerpt="{doc_row["excerpt"]}"')
    topic_notes_lines.append('')

topic_notes_lines.append('## 3. Bad topics')
topic_notes_lines.append('')
for key, info in BAD_TOPICS.items():
    model_name, topic_id = key.split('::')
    topic_id = int(topic_id)
    words = topic_words_all[(topic_words_all['model'] == model_name) & (topic_words_all['topic_id'] == topic_id)]['top_words'].iloc[0]
    topic_notes_lines.append(f'### {key}')
    topic_notes_lines.append(f'- Top words: {words}')
    topic_notes_lines.append(f'- Problem: {info["problem"]}')
    topic_notes_lines.append(f'- Why it is weak: {info["why"]}')
    topic_notes_lines.append(f'- What to change next: {info["next_fix"]}')
    topic_notes_lines.append('')

topic_notes_lines.append('## 4. LSA vs LDA comparison')
topic_notes_lines.append('')
topic_notes_lines.append(comparison_text)
topic_notes_lines.append('')
topic_notes_lines.append('## 5. Conclusion')
topic_notes_lines.append('')
topic_notes_lines.append('Topic modeling is useful here for corpus exploration, but the corpus is too mixed for uniformly clean topics. LDA is the safer default for this dataset, while LSA is more prone to duplicate and style-heavy topics.')
(docs_dir / 'topic_notes_lab8.md').write_text('\n'.join(topic_notes_lines), encoding='utf-8')

dataset_card_md = f'''# Dataset card — Lab8

## Corpus snapshot
- Source corpus: cleaned `processed_v2` from Lab2.
- Documents before filtering: {len(raw_df)}.
- Documents after topic-model filtering: {len(corpus_df)}.
- Filtering rule: keep documents with at least 4 cleaned tokens after stop-word and template-noise removal.

## Topic modeling findings
- Strong recurring themes: `єВідновлення / Дія`, `освітні заклади / викладачі`, `ратуша / вид на місто`.
- The corpus is mixed rather than homogeneous: it blends service reviews, university/academy reviews, tourism/location comments, and civic / war-support posts.
- Some noisy or template-style documents remain, especially short recommendation posts and generic service complaints.
- Topic modeling is useful for exploratory analysis here, but not strong enough on its own for stable domain segmentation.

## Remaining risks
- Mixed domains in one corpus create blended topics.
- Short texts still produce generic themes.
- Template phrases and service vocabulary can dominate weaker topics.
- A lemma-based representation may shift topic boundaries.
'''
(docs_dir / 'dataset_card.md').write_text(dataset_card_md, encoding='utf-8')

readme_md = f'''# LPNU NLP — Lab 08 (Topic Modeling: LSA vs LDA)

1. Corpus analyzed: cleaned Ukrainian review/comment corpus from `processed_v2`.
2. Models run: `LSA = TF-IDF + TruncatedSVD`, `LDA = CountVectorizer + LatentDirichletAllocation`.
3. Topic counts tested: `k=5` and `k=8` for both models.
4. Best themes: `єВідновлення / Дія`, `освітні заклади / навчання / викладачі`, `ратуша / вид на місто`.
5. Bad themes: duplicate LSA sightseeing/style topics and mixed LDA service-war topics.
6. LSA vs LDA: LDA produced more readable topics for this corpus because its top documents stayed more coherent.
7. Usefulness: topic modeling is helpful for corpus exploration, but the corpus should be split into narrower domains for cleaner topics.
'''
(ROOT / 'labs' / 'lab08' / 'README.md').write_text(readme_md, encoding='utf-8')

print((docs_dir / 'audit_summary_lab8.md').read_text(encoding='utf-8'))

# Audit summary — Lab8

1. Corpus size after filtering: 988 documents (from 1000 original rows).
2. Models tested: LSA (TF-IDF + TruncatedSVD) and LDA (CountVectorizer + LatentDirichletAllocation).
3. Topic counts tested: k=5 and k=8 for both models.
4. Best themes: єВідновлення / документи / цифрові сервіси; Ратуша / вид на місто / туристична рекомендація; Освітні заклади / навчання / викладачі.
5. Worst themes: duplicate sightseeing/style topic; generic opinion/place topic; mixed war/service topic.
6. What damaged the weak topics: template-style recommendation phrases, generic service vocabulary, and a mixed multi-domain corpus with many short documents.
7. Better model for this corpus: LDA, because its strongest topics stay more coherent at the document level.
8. Next steps: stronger stop-word filtering, domain-specific subcorpora, and optional lemma-based topic modeling.

